In [17]:
from task_maker import create_task, gen_locations, load_file
import pandas as pd
from itertools import product
import numpy as np

In [18]:
filename = "task_template.json"

In [19]:
num_tasks = 500

dataset = []
for _ in range(num_tasks):
    task = create_task(filename)
    dataset.append(task)
df = pd.DataFrame(dataset)

In [20]:
df.head()

,red_cube_start,blue_cube_start,yellow_cube_start,green_cube_start,Text Question,block_for_task,cat,other_blocks,red_cube_goal,blue_cube_goal,yellow_cube_goal,green_cube_goal
0,"(0.0, 0.4, 0.02)","(-0.3, 0.55, 0.02)","(0.15, 0.4, 0.02)","(-0.3, 0.25, 0.02)",Take the block that is coloured scarlet and place it at [0.15 0.25],0,0,None,"(0.15, 0.25, 0.02)","(-0.3, 0.55, 0.02)","(0.15, 0.4, 0.02)","(-0.3, 0.25, 0.02)"
1,"(0.0, 0.55, 0.02)","(0.15, 0.4, 0.02)","(-0.15, 0.55, 0.02)","(-0.15, 0.25, 0.02)","Take the block that is (-0.29999999999999993, 0.15000000000000002) from the blue block and place it between the blocks that are coloured red and rightmost",2,2,"[0, 1]","(0.0, 0.55, 0.02)","(0.15, 0.4, 0.02)","(0.3, 0.55, 0.02)","(-0.15, 0.25, 0.02)"
2,"(0.0, 0.25, 0.02)","(0.3, 0.4, 0.02)","(0.15, 0.25, 0.02)","(-0.15, 0.55, 0.02)","Take the block that is (-0.3, -0.15000000000000002) from the blue block and place it between the blocks that are coloured yellow and coloured green",0,2,"[2, 3]","(-0.0, 0.4, 0.02)","(0.3, 0.4, 0.02)","(0.15, 0.25, 0.02)","(-0.15, 0.55, 0.02)"
3,"(0.15, 0.4, 0.02)","(0.0, 0.55, 0.02)","(0.15, 0.55, 0.02)","(0.0, 0.25, 0.02)",Take the block that is coloured emerald and place it at [-0.3 0.25],3,0,None,"(0.15, 0.4, 0.02)","(0.0, 0.55, 0.02)","(0.15, 0.55, 0.02)","(-0.3, 0.25, 0.02)"
4,"(0.3, 0.4, 0.02)","(-0.15, 0.55, 0.02)","(-0.3, 0.55, 0.02)","(-0.3, 0.4, 0.02)",Take the block that is rightmost and place it at [0.15 0.55],0,0,None,"(0.15, 0.55, 0.02)","(-0.15, 0.55, 0.02)","(-0.3, 0.55, 0.02)","(-0.3, 0.4, 0.02)"


In [21]:
def is_valid(x):
    
    end_locs = (np.array(x['red_cube_goal']), 
                np.array(x['blue_cube_goal']), 
                np.array(x['green_cube_goal']), 
                np.array(x['yellow_cube_goal']))
    indices = list(range(4))
    dists = []
    for a, b in product(indices, repeat=2):
        if a != b:
            dists.append(np.linalg.norm(end_locs[a] - end_locs[b]))
    return min(dists)


template = load_file(filename)
_, _, valid_locations, _ = gen_locations(template['location_grid'])

def in_ok_spot(x):
    end_locs = (np.array(x['red_cube_goal']), 
                np.array(x['blue_cube_goal']), 
                np.array(x['green_cube_goal']), 
                np.array(x['yellow_cube_goal']))
    for loc in end_locs:
        for loc_pr in valid_locations:
            if np.linalg.norm(loc[:2]-loc_pr) < 0.1:
                return True
    return False

In [22]:
df['goal_dist'] = df.apply(is_valid, axis=1)
df['all_in_grid'] = df.apply(in_ok_spot, axis=1)

In [23]:
pd.set_option('display.max_colwidth', None)

In [24]:
df[df['goal_dist'] <= 0.1]#[['Text Question'] + [i for i in df.columns.tolist() if 'goal' in i]]

,red_cube_start,blue_cube_start,yellow_cube_start,green_cube_start,Text Question,block_for_task,cat,other_blocks,red_cube_goal,blue_cube_goal,yellow_cube_goal,green_cube_goal,goal_dist,all_in_grid


In [25]:
df[df['all_in_grid'] == False]

,red_cube_start,blue_cube_start,yellow_cube_start,green_cube_start,Text Question,block_for_task,cat,other_blocks,red_cube_goal,blue_cube_goal,yellow_cube_goal,green_cube_goal,goal_dist,all_in_grid


In [26]:
df = df[[c for c in df.columns.tolist() if c not in ['goal_dist','all_in_grid']]]
df.to_csv("task_dataset.csv", index=False)

In [27]:
data = pd.read_csv('task_dataset.csv')

In [28]:
task_observations = []
for rs,bs,gs,ys in zip(data['red_cube_start'],data['blue_cube_start'],data['green_cube_start'],data['yellow_cube_start']):
    return_str = '' 
    return_str += '*** Observation *** \n'
    return_str +=  'Here is the initial position of each block and the end effector: \n'
    return_str += 'Red Cube: ' + str(eval(rs)[0:2]) + '\n'
    return_str += 'Green Cube: ' + str(eval(gs)[0:2]) + '\n'
    return_str += 'Blue Cube: ' + str(eval(bs)[0:2]) + '\n'
    return_str += 'Yellow Cube: ' + str(eval(ys)[0:2]) + '\n'
    return_str += 'End Effector: ' + '(0, 0.25)' + '\n'
    return_str += 'Note - the arm itself is at 0,0. Assume that the x-axis determines left and right and the y-axis top and bottom.'
    task_observations.append(return_str)
print(len(task_observations))
print(task_observations[0])

500
*** Observation *** 
Here is the initial position of each block and the end effector: 
Red Cube: (0.0, 0.4)
Green Cube: (-0.3, 0.25)
Blue Cube: (-0.3, 0.55)
Yellow Cube: (0.15, 0.4)
End Effector: (0, 0.25)
Note - the arm itself is at 0,0. Assume that the x-axis determines left and right and the y-axis top and bottom.


In [29]:
# sanity check
data['observation'] = task_observations
print(data.iloc[0])
data.iloc[0]['observation']

red_cube_start                                                                                                                                                                                                                                                                                                                            (0.0, 0.4, 0.02)
blue_cube_start                                                                                                                                                                                                                                                                                                                         (-0.3, 0.55, 0.02)
yellow_cube_start                                                                                                                                                                                                                                                                                                 

'*** Observation *** \nHere is the initial position of each block and the end effector: \nRed Cube: (0.0, 0.4)\nGreen Cube: (-0.3, 0.25)\nBlue Cube: (-0.3, 0.55)\nYellow Cube: (0.15, 0.4)\nEnd Effector: (0, 0.25)\nNote - the arm itself is at 0,0. Assume that the x-axis determines left and right and the y-axis top and bottom.'

In [30]:
# saves
data.to_csv('task_dataset.csv')